In [18]:
%load_ext rich
%load_ext autoreload
%autoreload 2


The rich extension is already loaded. To reload it, use:
  %reload_ext rich
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


[autoreload of scripts.core failed: Traceback (most recent call last):
  File "/lustre/work/client/users/stevensonb/.conda/envs/mlpng/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/lustre/work/client/users/stevensonb/.conda/envs/mlpng/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 475, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "/lustre/work/client/users/stevensonb/.conda/envs/mlpng/lib/python3.11/importlib/__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 621, in _exec
  File "<frozen importlib._bootstrap_external>", line 940, in exec_module
  File "<frozen importlib._bootstrap>", line 241, in _call_with_frames_removed
  File "/users/stevensonb/Research/MLPNG/scripts/core.py", line 16, in <module>
    from utils import remove_mono_dipole
ModuleNotFoundError: No module named 'ut

In [19]:
import logging
import os
import sys

from rich.jupyter import print

import camb
import healpy as hp
import numpy as np
from joblib import Parallel, delayed
from ksw import Cosmology, Data, Shape, KSW
from ksw.radial_functional import radial_func
from scipy.interpolate import CubicSpline
from tqdm.auto import tqdm
from scripts.core import Core
from scripts.utils import (
    load_data,
    save_data,
    setup_logging,
    remove_mono_dipole,
)
from scripts.utils.plots import plot_cl_alm, plot_cl_map, plot_patches, plot_cl
from itertools import product

from functools import partial

import lenspyx
import matplotlib.pyplot as plt  # type: ignore

from pixell import curvedsky, enmap, reproject

logger = setup_logging(__name__, level=logging.DEBUG, use_rich=True)
logging.getLogger("matplotlib").setLevel(logging.WARNING)


## Setup

In [20]:
%%time
s = Core(
    [
        "settings/planck.json",
        "--nsims",
        "1",
        "--narray",
        "1",
    ],
    alm=True,
    patch=True,
    # estimator=True,
)
s.is_main_job = True  # fix an issue with missing slurm job array


16-Apr-24 12:38:33 - scripts.core - DEBUG - parsing cli args: ['settings/planck.json', '--nsims', '1', '--narray', 
'1']

16-Apr-24 12:38:33 - scripts.core - INFO - Loading settings from file settings/planck.json

16-Apr-24 12:38:33 - scripts.core - DEBUG - Forcing setting 'nsims' to '1' due to CLI

16-Apr-24 12:38:33 - scripts.core - DEBUG - Forcing setting 'narray' to '1' due to CLI

16-Apr-24 12:38:33 - scripts.core - DEBUG - Found non-default value for 'cosmo_params': {'H0': 70.1, 'As':         
2.457e-09, 'ns': 0.96, 'ombh2': 0.02256, 'omch2': 0.1143, 'tau': 0.084, 'max_l': 1500, 'lmax': 1024} (default: {})

...

16-Apr-24 12:38:33 - scripts.core - DEBUG - Setting 'alm_dir' not found, using default: 'alms'

16-Apr-24 12:38:33 - scripts.core - DEBUG - Setting 'plot_dir' not found, using default: 'plots'

16-Apr-24 12:38:33 - scripts.core - DEBUG - Setting 'tb_dir' not found, using default: 'tensorboard'

16-Apr-24 12:38:33 - scripts.core - DEBUG - Setting 'model_dir' not found, using default: 'models'

16-Apr-24 12:38:33 - scripts.core - DEBUG - Found non-default value for 'base_name': 'planck_512' (default:        
'l1024_n512_ul_T_1')

## Alm

In [ ]:
def get_alm(alm, bl_div_cl, alpha_l, r, dr, nside, lmax):
    """This calculates the alms from the precalculated values"""
    Balm = hp.almxfl(alm, bl_div_cl)
    B = hp.alm2map(Balm, nside=nside, lmax=lmax)
    inner = hp.map2alm(B**2, lmax=lmax, use_pixel_weights=True)
    kernel = hp.almxfl(inner, alpha_l)
    return dr * r**2 * kernel


def interpolate_ells(func, ells_sparse, ls, axis=1):
    """Our interpolation function to go from space to dense ells."""
    data = CubicSpline(ells_sparse, func, axis)(ls)
    logger.info(f"Interpolated data shape: {data.shape}")
    return data


def generate_almngs(s, alms):
    """This code completely calculates, and saves, the alms and almngs."""
    # $$a_{\ell m}^{NG,loc'} = \int dr r^2 \left[ \alpha_\ell(r)\left(\int d^2 \hat{n} Y_{\ell m}^\star (\hat{n}) B(r,\hat{n})^2 \right)\right]$$
    # and
    # $$\alpha_\ell(r)=\frac{2}{\pi} \int_0^\infty dk k^2 \Delta_\ell^T(k) j_\ell(k r)$$
    # $$\beta_\ell(r)=\frac{2}{\pi} \int_0^\infty dk k^{-1} \Delta_\phi \Delta_\ell^T(k) j_\ell(k r)$$
    # $$B(r, \hat{n}) = \sum_{\ell,m} \frac{\beta_\ell (r)}{C_\ell} a_{\ell m} Y_{\ell m}$$
    # where $\Delta_\phi$ is primordial normalization, $\Delta_\ell^T(k)$ is the transfer function, $j_\ell(k r)$ are the spherical bessel functions
    A = (3 / 5) ** 2 * 2 * np.pi**2 * s.cosmo_params["As"]
    delta_phi = (s.tr_k) ** ((s.cosmo_params["ns"] - 1)) / (s.tr_k**3)

    f_k = np.ones((len(s.tr_k), 2), dtype=s.r_dtype) * 5 / 3
    # f_k[:, 0] = 1             # f_k for alpha
    f_k[:, 1] *= A * delta_phi  # f_k for beta

    rad = radial_func(f_k, s.tr_ell_k, s.tr_k, s.radii, s.tr_ells)

    alpha_ell = rad[:, :, :, 0]
    alpha_l = interpolate_ells(alpha_ell, s.tr_ells, s.ells)
    logger.info(f"Alpha_l shape: {alpha_l.shape}")
    beta_ell = rad[:, :, :, 1]
    c_ells_new = s.c_ells["c_ell"][s.tr_ells, : s.npol]
    div = beta_ell / c_ells_new[np.newaxis, :, :]
    bl_div_cl = interpolate_ells(div, s.tr_ells, s.ells)
    bl_div_cl = np.concatenate(np.array([bl_div_cl]))
    logger.info(f"Bl_div_cl shape: {bl_div_cl.shape}")

    # We use joblib.parallel to generate the patches in parallel
    # by default (temp_folder=None) this will use a ram disk /dev/shm
    # if the data files are larger than the available memory, it will error
    # so we give it a temp folder to use, which wont have that problem
    logger.info("Starting non-gaussian Alm generation")
    temp_folder = os.environ.get("SCRATCH", None)
    logger.debug(f"Using temp folder for almng generation: {temp_folder}")
    parallel = Parallel(n_jobs=-1, return_as="generator", temp_folder=temp_folder)
    almng = np.empty(s.alm_shape, dtype=s.c_dtype)
    for i, pol in tqdm(s.sim_pol, total=s.sim_pol_len, desc="Almng"):
        logger.debug("Generating almng[%d,%d]", i, pol)
        generator = parallel(
            delayed(get_alm)(
                alms[i, pol],
                bl_div_cl[ri, :, pol],
                alpha_l[ri, :, pol],
                s.radii[ri],
                s.drs[ri],
                s.nside,
                s.lmax,
            )
            for ri in range(len(s.drs))
        )

        almng[i, pol] = sum(generator)
        logger.debug("Almng[%d,%d] shape: %s", i, pol, almng[i, pol].shape)
    logger.info("Non-gaussian Alm generation complete")
    return almng


This code generates the alms

$$a_{\ell m} = a_{\ell m}^{{G}} + f_{NL}^X a_{\ell m}^{NG}$$
with
$$a_{\ell m}^{NG,loc'} = \int dr r^2 \left[ \alpha_\ell(r)\left(\int d^2 \hat{n} Y_{\ell m}^\star (\hat{n}) B(r,\hat{n})^2 \right)\right]$$
and
$$\alpha_\ell(r)=\frac{2}{\pi} \int_0^\infty dk k^2 \Delta_\ell^T(k) j_\ell(k r)$$
$$\beta_\ell(r)=\frac{2}{\pi} \int_0^\infty dk k^{-1} \Delta_\phi \Delta_\ell^T(k) j_\ell(k r)$$
$$B(r, \hat{n}) = \sum_{\ell,m} \frac{\beta_\ell (r)}{C_\ell} a_{\ell m} Y_{\ell m}$$
where $\Delta_\phi$ is primordial normalization, $\Delta_\ell^T(k)$ is the transfer function, $j_\ell(k r)$ are the spherical bessel functions   

In [ ]:
# Get our alms
logger.info("Starting gaussian Alm generation")
alm_l = np.array(
    [s.data.compute_alm_sim(s.lensing) for _ in range(s.nsims)],
    dtype=s.c_dtype,
)

alm_ng = generate_almngs(s, alm_l)
fnls = s.rng.uniform(s.fnl_min, s.fnl_max + 1, s.fnl_shape)


16-Apr-24 12:30:54 - __main__ - INFO - Starting gaussian Alm generation

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:4                                                                                    │
│                                                                                                  │
│    1 # Get our alms                                                                              │
│    2 logger.info("Starting gaussian Alm generation")                                             │
│    3 alm_l = np.array(                                                                           │
│ ❱  4 │   [s.data.compute_alm_sim(s.lensing) for _ in range(s.nsims)],                            │
│    5 │   dtype=s.c_dtype,                                                                        │
│    6 )                                                                                           │
│    7                                                                                             │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
NameError: name 's' is not defined

In [ ]:
alms = alm_l + fnls * alm_ng

if s.noise:
    for i, j in s.sim_pol:
        hp.almxfl(alms[i, j], s.beam_ell[j] ** -1, inplace=True)
        alms[i, j] += hp.synalm(s.noise_ell[j], lmax=s.lmax, new=True)

alms = remove_mono_dipole(alms)


In [ ]:
i, j = s.rng.integers(s.nsims), s.rng.integers(s.npol)

plot_cl_alm(alms[i, j], plot_camb=True, c_ells=s.c_ells)


## PatchGen

In [ ]:
def cutSqPatches_lenspyx(
    lmax,
    max_l,
    r_dtype,
    npatches,
    nside,
    fs_shape,
    fs_wcs,
    fs_map,
    pshapes,
    pwcs,
    cl_phi,
    alm,
    fnl,
    plot=False,
):
    """Uses lenspyx to generate and cut the lensed flat maps"""
    geom_info = ("healpix", {"nside": nside})

    # Create a full sky map with lenspyx
    plm = lenspyx.utils_hp.synalm(cl_phi, lmax=max_l, mmax=None)
    fl = np.sqrt(np.arange(max_l + 1) * np.arange(1, max_l + 2), dtype=r_dtype)
    dlm = lenspyx.utils_hp.almxfl(
        plm, fl, mmax=None, inplace=False
    )  # inplace breaks, for some reason

    lens_map = lenspyx.alm2lenmap(
        alm.copy(),
        dlm,
        geometry=geom_info,
        nthreads=4,
        epsilon=1e-6,
        pol=False,
    )
    pixell_map = reproject.healpix2map(lens_map, fs_shape, fs_wcs, lmax)

    if plot:
        map2hp = reproject.map2healpix(pixell_map, lmax)
        hp.mollview(map2hp, min=-650.0, max=650, title=f"fnl = {fnl}")
        plt.show()
        plot_cl(hp.anafast(map2hp), lmax)

    patches = []
    for i in range(npatches):
        patch = pixell_map.project(pshapes[i], pwcs[i])
        patches.append(patch)

    return patches


def cutSqPatches_pixell(
    lmax,
    plot_dir,
    base_name,
    npatches,
    c_ells,
    fs_shape,
    fs_wcs,
    fs_map,
    pshapes,
    pwcs,
    alms,
    fnl,
    plot=False,
):
    """Uses pixell to generate and cut the flat sky patches, unlensed"""
    fs_map = enmap.empty(fs_shape, fs_wcs)
    car_map = curvedsky.alm2map(alms, fs_map)

    patches = []
    for i in range(npatches):
        patch = car_map.project(pshapes[i], pwcs[i])  # type: ignore
        patches.append(patch)

    if plot:
        plot_dir = os.path.join(plot_dir, "patchgen")
        os.makedirs(plot_dir, exist_ok=True)

        map2hp = reproject.map2healpix(car_map, lmax)
        hp.mollview(map2hp, min=-650.0, max=650, title=f"fnl = {fnl}")
        plt.savefig(os.path.join(plot_dir, f"{base_name}_{fnl}_fullsky.png"))
        plt.close()

        map_path = os.path.join(plot_dir, f"{base_name}_{fnl}_pixell_cl_map.png")
        plot_cl_map(
            car_map, fs_wcs, lmax, plot_camb=True, c_ells=c_ells, save_file=map_path
        )

        patch_path = os.path.join(plot_dir, f"{base_name}_{fnl}_patch.png")
        plot_patches(np.array(patches), 8, save_file=patch_path)

    return np.array(patches)


def get_fs_patch_geo():
    """Generates the patch geometry using pixell"""
    res = np.deg2rad(s.patch_side_deg / s.nside)
    ps_rad = np.deg2rad(s.patch_side_deg)
    fs_shape, fs_wcs = enmap.fullsky_geometry(res, proj="car")
    fs_map = enmap.empty(fs_shape, fs_wcs)

    patch_shapes = []
    patch_wcss = []
    for counter in np.arange(s.npatches // 2):
        # [[dec_min,ra_min],[dec_max,ra_max]]
        top = [[0, ps_rad * counter], [ps_rad, ps_rad * (counter + 1)]]
        gs, w = enmap.geometry(pos=top, res=res, proj="car")
        patch_shapes.append(gs)
        patch_wcss.append(w)

        bottom = [[-ps_rad, ps_rad * counter], [0, ps_rad * (counter + 1)]]
        gs, w = enmap.geometry(pos=bottom, res=res, proj="car")
        patch_shapes.append(gs)
        patch_wcss.append(w)
    return fs_shape, fs_wcs, fs_map, patch_shapes, patch_wcss


In [ ]:
# Here we get the geometry of our patches in a tuple
patch_geo = get_fs_patch_geo()

# we also setup the cutPatches function to use either pixell or lenspyx
# depending on if we are doing lensing or not, and provide a lot of
# arguments that are needed and will stay constant
# we cannot abuse the python scope here since these will need to be pickled
if s.lensing:
    logger.debug("Using lenspyx to generate patches")
    cl_phi = s.cosmo._camb_data.get_lens_potential_cls(
        s.cosmo_params["max_l"], CMB_unit="muK", raw_cl=True
    )[:, 0]

    cutPatches = partial(
        cutSqPatches_lenspyx,
        s.lmax,
        s.cosmo_params["max_l"],
        s.r_dtype,
        s.npatches,
        s.nside,
        *patch_geo,
        cl_phi,
    )
else:
    logger.debug("Using pixell to generate patches")
    cutPatches = partial(
        cutSqPatches_pixell,
        s.lmax,
        s.plot_dir,
        s.base_name,
        s.npatches,
        s.c_ells,
        *patch_geo,
    )

## Start the patch generation
# create the array to store the patches
patches = np.empty(s.patch_shape, dtype=s.r_dtype)

# We use joblib.parallel to generate the patches in parallel
# by default (temp_folder=None) this will use a ram disk /dev/shm
# if the data files are larger than the available memory, about 1TB, it will error
# so we give it a temp folder to use, which wont have that problem
temp_folder = os.environ.get("SCRATCH", None)
logger.debug(f"Using temp folder for patch generation: {temp_folder}")
patch_generator = Parallel(
    n_jobs=1,
    return_as="generator",
    temp_folder=temp_folder,
)(delayed(cutPatches)(alms[i, j], fnls[i, j], True) for i, j in s.sim_pol)

# Get our data from the generator, only update logging every 100 runs, takes a long time
for idx, result in enumerate(
    tqdm(patch_generator, desc="patch progress", total=s.sim_pol_len)
):
    i, pol = s.sim_pol[idx]
    patches[i, pol] = result


In [ ]:
plot_patches(patches[0, 0], 10)


## Estimator

In [ ]:
mc_file = os.path.join(s.alm_dir, "kswmc", f"{s.base_name}_m.hdf5")
if os.path.exists(mc_file):
    logger.info("Loading KSW state from %s", mc_file)
    s.ksw.start_from_read_state(mc_file)
else:
    raise NotImplementedError(
        f"No MC file, {mc_file}, found. This notebook requires more work"
    )

fisher = float(s.ksw.compute_fisher())
logger.info("Fisher: %s, standard deviation: %s", fisher, np.sqrt(1 / fisher))


def alm_loader(i):
    return alms[i, 0]


idxs = [0]
estimates = s.ksw.compute_estimate_batch(alm_loader, idxs, fisher=fisher)
